# Lab 05: The Dress That Broke the Internet

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 1, *The Reality Generator*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-05.ipynb)

**What you will do:** record your own colour percept for the Adelson checkerboard and #TheDress, compare it with the published categories, explore a model of Bayesian colour correction under different assumed light sources, and test whether a classical auto-white-balance algorithm resolves the same ambiguity to a single answer.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 05 you isolated two squares of the Adelson checkerboard (a cylinder casting a shadow
across a green-and-white grid) using punched holes or your fingers, and found they are the same
shade even though context makes them look different. You then looked at #TheDress and recorded
whether you saw it as white and gold or blue and black. Finally, you made (or imagined) a
coloured shadow: a patch that looks tinted even though it is really just a darker region of
white surface.

## Record your results

Log your own results from Steps 1-2: whether covering the Adelson checkerboard's context (with
punched holes or your fingers) changed how different the two squares looked, and which colour
category you saw for #TheDress, with your confidence.

In [ ]:
# example data: replace with your own observations
dress_report = pd.DataFrame({
    "test": ["Adelson squares, in context", "Adelson squares, isolated", "#TheDress"],
    "your_report": ["look different shades", "look the same shade", "white and gold"],  # example data
    "confidence_1to5": [4, 5, 4],   # example data
})
display(dress_report)

## Compare

Lafer-Sousa, Hermann, and Conway (2015) surveyed over a thousand people about #TheDress and
found perception clustered into three groups: most people reported either white/gold or
blue/black, with a smaller number reporting an intermediate blue/brown. Compare your own
report with these published categories, and check your Adelson result against what the test is
designed to show (identical squares, different apparent shade, purely from context).

In [ ]:
dress_answer = dress_report.loc[dress_report.test == "#TheDress", "your_report"].iloc[0]
print(f"Your #TheDress report: {dress_answer}")
if dress_answer.lower() in ("white and gold", "blue and black"):
    print("You fall into one of the two large groups Lafer-Sousa et al. (2015) found.")
else:
    print("You fall into the smaller 'intermediate' group the study also reported (or gave a")
    print("different description) -- still consistent with the study's finding that perception")
    print("clusters rather than being random.")

isolated = dress_report.loc[dress_report.test == "Adelson squares, isolated", "your_report"].iloc[0]
if "same" in isolated.lower():
    print("\nYour Adelson result matches the book: once you isolate the squares from their")
    print("context, the illusion collapses and they look the same shade.")
else:
    print("\nYour Adelson result does not match the book's expected collapse -- try isolating")
    print("the squares again with a smaller viewing hole that excludes the shadow's edge.")

## Simulation

Your brain saw #TheDress as blue-black or white-gold depending on its assumption about the light source. This simulation illustrates how two different priors (bluish daylight vs. warm indoor light) applied to the same pixel data produce two completely different colour percepts.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# The ambiguous dress pixel (approximate RGB from the photograph)
dress_pixel = np.array([100, 120, 180])

# Two plausible assumptions about the illumination
prior_shadow = np.array([180, 200, 255])   # cool, bluish (shade or daylight)
prior_indoor = np.array([255, 210, 150])   # warm, yellowish (tungsten)

# The brain discounts the assumed illuminant to estimate the surface colour
surface_if_shadow = np.clip(dress_pixel / (prior_shadow / 255), 0,
    255).astype(int)
surface_if_indoor = np.clip(dress_pixel / (prior_indoor / 255), 0,
    255).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(10, 3))
for ax, colour, title in zip(axes,
    [dress_pixel, surface_if_shadow, surface_if_indoor],
    ["Raw pixel", "If shade/daylight assumed\n→ white/gold",
        "If indoor tungsten assumed\n→ blue/black"]):
    ax.imshow([[colour / 255]])
    ax.set_title(title, fontsize=10)
    ax.axis("off")
    ax.text(0.5, -0.15, f"RGB {tuple(colour)}", ha="center",
        transform=ax.transAxes, fontsize=8)
plt.suptitle("Same pixel, different prior about the light source",
    fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

**What to notice:** The raw pixel is identical in both cases. Discounting an assumed bluish light leaves a paler, warmer residual (read as white/gold); discounting an assumed warm light leaves a residual dominated by blue (read as blue/black). The only difference is the brain's assumption about the light. This is Bayesian perception in action—change the prior, change the reality.

## Change the parameters

Set the raw pixel's red, green and blue values with the sliders, then choose which light
source the brain assumes with the dropdown. The two swatches show the raw pixel and the
surface colour the brain recovers once it discounts that assumed illuminant.
(If the sliders do not appear, run the cell again.)

In [ ]:
from ipywidgets import interact, IntSlider, Dropdown

def correct(pixel, prior):
    return np.clip(pixel / (prior / 255), 0, 255).astype(int)

def explore(r=100, g=120, b=180, illuminant="Cool shade (bluish)"):
    pixel = np.array([r, g, b])
    prior = prior_shadow if illuminant.startswith("Cool") else prior_indoor
    surface = correct(pixel, prior)

    fig, axes = plt.subplots(1, 2, figsize=(6, 3))
    axes[0].imshow([[pixel / 255]])
    axes[0].set_title(f"Raw pixel\nRGB {tuple(pixel)}", fontsize=10)
    axes[0].axis("off")
    axes[1].imshow([[surface / 255]])
    axes[1].set_title(f"Assumed: {illuminant}\nCorrected RGB {tuple(surface)}", fontsize=10)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

interact(explore,
         r=IntSlider(value=100, min=0, max=255, step=5),
         g=IntSlider(value=120, min=0, max=255, step=5),
         b=IntSlider(value=180, min=0, max=255, step=5),
         illuminant=Dropdown(options=["Cool shade (bluish)", "Warm indoor (tungsten)"]));

## The AI side

Smartphone cameras must resolve the same ambiguity that produced #TheDress: they estimate the
scene's illumination (**auto white balance**) and correct for it before showing you a
photograph. Below, OpenCV's classical grey-world white-balance algorithm -- which assumes the
average colour in a scene should be neutral grey -- is given the same ambiguous patch from the
simulation above, once surrounded by a "cool" scene and once by a "warm" scene. Unlike you, the
algorithm cannot hold two answers at once: watch whether changing only the surrounding context
changes its single, confident correction.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q opencv-contrib-python
import cv2

def scene(background_tint):
    img = np.full((120, 120, 3), background_tint, dtype=np.uint8)
    img[40:80, 40:80] = dress_pixel.astype(np.uint8)  # the ambiguous patch from the simulation above
    return img

cool_scene = scene((190, 205, 230))   # example data: a scene dominated by cool, bluish light
warm_scene = scene((235, 205, 165))   # example data: a scene dominated by warm, yellowish light

wb = cv2.xphoto.createGrayworldWB()
corrected_cool = wb.balanceWhite(cool_scene)
corrected_warm = wb.balanceWhite(warm_scene)

fig, axes = plt.subplots(2, 2, figsize=(5, 5))
for ax, img, title in zip(axes.flat,
                           [cool_scene, corrected_cool, warm_scene, corrected_warm],
                           ["Cool scene (raw)", "Cool scene (white-balanced)",
                            "Warm scene (raw)", "Warm scene (white-balanced)"]):
    ax.imshow(img); ax.set_title(title, fontsize=9); ax.axis("off")
plt.tight_layout(); plt.show()

patch_cool = corrected_cool[55:65, 55:65].reshape(-1, 3).mean(axis=0)
patch_warm = corrected_warm[55:65, 55:65].reshape(-1, 3).mean(axis=0)
print(f"Patch after cool-scene correction: RGB {tuple(patch_cool.round(0))}")
print(f"Patch after warm-scene correction: RGB {tuple(patch_warm.round(0))}")

The identical ambiguous patch comes out with two different corrected colours, depending only
on the scene around it -- the algorithmic equivalent of Wallisch's (2017) finding that
assumptions about the surrounding light, not the pixel itself, decide what colour people see.
The disanalogy: grey-world white balance always commits to exactly one answer, computed from a
simple statistical rule (the scene should average out to grey). Your brain instead holds a
strong, specific prior about likely light sources (daylight, tungsten) built from a lifetime of
experience, which is why two people, both with the same "grey-world"-style statistics
available, can still land on stubbornly different answers.

## Pool the class data

Pool the class's #TheDress reports: each person adds an anonymous ID and their colour category,
downloads the results as CSV and loads it here.

In [ ]:
import io
csv_text = """id,dress_report
P01,white and gold
P02,blue and black
P03,blue and black
P04,white and gold
P05,blue and brown
"""  # example data: replace with your class CSV, e.g. pd.read_csv("dress_class.csv")
df = pd.read_csv(io.StringIO(csv_text))
counts = df.dress_report.value_counts()
display(counts)

boot_props = []
for _ in range(5000):
    sample = df.dress_report.sample(len(df), replace=True)
    boot_props.append(sample.value_counts(normalize=True))
boot_df = pd.DataFrame(boot_props).fillna(0)
props = counts / len(df)
print("Proportion reporting each category (95% bootstrap interval):")
for cat in props.index:
    lo_ci, hi_ci = np.percentile(boot_df[cat], [2.5, 97.5])
    print(f"  {cat}: {props[cat]:.0%} ({lo_ci:.0%}-{hi_ci:.0%}, n = {len(df)})")

counts.plot.bar(color="steelblue", figsize=(6, 3))
plt.ylabel("number of people"); plt.xticks(rotation=20, ha="right"); plt.tight_layout(); plt.show()

## Questions to think about

1. Keep the pixel at its default RGB values and switch the illuminant dropdown between "Cool shade" and "Warm indoor". Which choice produces white and gold, and which produces blue and black, and does either match how you personally see the dress photograph?
2. Keep the illuminant fixed on "Cool shade" and change only the blue slider (b) between 100 and 255. Does the corrected surface colour change a lot, or does the choice of illuminant dominate the outcome regardless of the raw pixel?
3. Wallisch (2017) found that people with more daytime exposure to natural light were more likely to assume a bluish illuminant and see white and gold. Using the widget, explain in your own words why that exposure history would push someone toward the "Cool shade" assumption rather than the "Warm indoor" one.
4. Set r, g and b to equal values, such as 150, 150 and 150, so the raw pixel is a neutral grey. Does switching the illuminant still produce two different colours? What does that tell you about whether the ambiguity lives in the pixel itself or in the brain's assumption about the light?

## Challenge

**Test your own colour constancy.** Try the coloured-shadow demonstration from Step 3 of Lab 05
(or find a photograph of a coloured shadow online). Estimate the approximate RGB values of the
shadow region from a photo, then use the code below -- which reuses the same `correct()` function
from the widget above -- to see what colour the two candidate illuminants imply, and compare that
with what your own eyes report.

In [ ]:
# Example data: replace with RGB values you estimate from your own coloured-shadow photo.
shadow_pixel = np.array([180, 190, 200])   # example: the shadowed region

# Two candidate assumptions about the ambient light that produced that shadow
prior_a = np.array([200, 210, 230])   # example: a cool ambient-light guess
prior_b = np.array([230, 200, 180])   # example: a warm ambient-light guess

print("If the light was cool:", tuple(correct(shadow_pixel, prior_a)))
print("If the light was warm:", tuple(correct(shadow_pixel, prior_b)))
print("The pixel and prior values above are example data -- replace them with your own")
print("estimates from a real photo, then compare the result with what you actually perceive.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-05.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")